# Scheduling: Resources, Affinity, Taints & Tolerations

## What's covered

- **The scheduling problem** — what `kube-scheduler` actually does, and what "binding" means
- **Resource requests and limits** — CPU and memory, the `m` / `Mi` / `Gi` units, and what happens when each is set or omitted
- **QoS classes** — `Guaranteed`, `Burstable`, `BestEffort`, and how they decide which pods get evicted first
- **`LimitRange` and `ResourceQuota`** — namespace-level defaults and hard caps
- **`nodeSelector`** — the simplest "put me on this kind of node" tool
- **Node affinity** — required and preferred, set-based selectors, and when to reach for them instead of `nodeSelector`
- **Pod affinity and anti-affinity** — placement relative to *other pods*, with the `topologyKey` trick
- **Taints and tolerations** — the node-side "go away unless you've explicitly opted in"
- **Eviction** — what the kubelet does under memory or disk pressure, and how QoS class drives the order
- **Topology spread constraints** — even distribution of replicas across zones, hostnames, or any topology
- **`PriorityClass` and preemption** — high-priority pods that can kick lower ones out
- **How the scheduler decides** — filtering, scoring, and scheduling profiles in one paragraph

By the end of this notebook you should be able to write a pod that's guaranteed not to be evicted under memory pressure, pin pods to specific nodes, repel pods from special-purpose nodes, and reason about why the scheduler made the choice it made. Notebooks 02 and 03 are the prerequisites — every scheduling field in this notebook lives in `spec` of a pod or a Deployment's pod template.

## The scheduling problem — what `kube-scheduler` does

When you `kubectl apply` a Deployment, the Deployment controller creates a ReplicaSet, the ReplicaSet controller creates Pods, and the API server writes them to `etcd` — with `spec.nodeName` empty. At this point the pods are `Pending`. **No node has been chosen.**

The **`kube-scheduler`** picks up there. It runs a tight loop:

1. **Watch** the API server for pods that have no `nodeName` set.
2. For each such pod, **filter** the cluster's nodes — keep only those that *could* run this pod (enough free CPU and memory, taints satisfied, node selectors satisfied, volumes available in the right zone, and so on).
3. **Score** the remaining nodes — assign a number from 0 to 100 reflecting how *good* a fit each one is.
4. Pick the highest-scoring node and **bind** the pod to it — write `spec.nodeName` via the API server's `bind` subresource.
5. The kubelet on that node sees `pod.spec.nodeName == <its own name>`, pulls the image, and starts the container.

```
        +---------------------+
        | API server          |
        |   Pods with         |
        |   nodeName == ""    |
        +----------+----------+
                   |  watch
                   v
        +---------------------+
        | kube-scheduler      |
        |  1. filter nodes    |
        |  2. score nodes     |
        |  3. bind pod        |
        +----------+----------+
                   |  PATCH spec.nodeName = <node>
                   v
        +---------------------+      +----------+
        | API server          | <--- | kubelet  |  sees its own name on the pod,
        |   bound pod object  | watch| (node)   |  starts the container
        +---------------------+      +----------+
```

A few things to internalise:

- **The scheduler doesn't run anything.** It just *decides* where the pod goes. The kubelet runs it.
- **The scheduler is a controller.** Same loop pattern as everything else in Kubernetes — read state, take action, write state.
- **The scheduler is pluggable.** A cluster can run more than one scheduler, or replace the default with a custom one. Most clusters don't.

This notebook is about the **inputs** that bias the scheduler's decision: how much you ask for, where you say you'll fit, where you say you won't fit, and where you say "spread out.

## Resource requests and limits

Every container can declare what it expects from the node. Two fields, both optional, both per-container:

```yaml
spec:
  containers:
    - name: app
      image: nginx
      resources:
        requests:
          cpu: 250m         # 250 millicores = 0.25 of one CPU
          memory: 256Mi     # 256 mebibytes
        limits:
          cpu: 500m
          memory: 512Mi
```

### What `requests` means

> **The minimum amount of resource the scheduler must reserve on the chosen node.**

When the scheduler filters nodes, it sums the `requests` of every pod already on the node and subtracts that from the node's allocatable capacity. A pod can only land on a node if its `requests` fit in the remaining headroom.

`requests` is **not** a runtime cap. A container with `requests.cpu: 250m` can use *more* CPU at runtime if the node has it free. `requests` is a scheduling input.

### What `limits` means

> **The maximum amount of resource the runtime will let the container actually use.**

Enforced by the kernel cgroup:

- **CPU limit** — the container is *throttled* if it tries to use more. No restart, just slow.
- **Memory limit** — if the container tries to allocate above the limit, the kernel **OOM-kills** the offending process. The container restarts (subject to `restartPolicy`).

### Units

- **CPU.** `1` means one whole CPU core (one hyperthread, really). `500m` (millicores) means 0.5 of one core. Sub-millicore is not allowed. Fractional decimals (`0.5`) are accepted and equivalent to `500m`.
- **Memory.** Default unit is bytes. Use binary suffixes — `Ki`, `Mi`, `Gi`, `Ti` (powers of 1024). Decimal suffixes (`K`, `M`, `G`, `T`, powers of 1000) also work but are unusual.

### What if you omit them?

- **`requests` omitted, `limits` set** — `requests` defaults to `limits`. (You get a "Guaranteed" pod — see below.)
- **`requests` set, `limits` omitted** — no upper bound. Can hog the node.
- **Both omitted** — the pod schedules onto any node with *any* free capacity, then runs unbounded. Dangerous in shared clusters. Fixed by `LimitRange` (covered shortly).

### Ephemeral storage

`ephemeral-storage` is a third resource type — it's the node's local writable scratch (`emptyDir`, container writable layer, logs). Set requests and limits on it the same way; the kubelet evicts pods that exceed.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: sized
spec:
  containers:
    - name: app
      image: nginx:alpine
      resources:
        requests:
          cpu: 100m
          memory: 64Mi
        limits:
          cpu: 200m
          memory: 128Mi
EOF
!sleep 5
!kubectl get pod sized
!echo '---'
# describe shows requests, limits, and the assigned QoS class
!kubectl describe pod sized | sed -n '/Containers:/,/Conditions:/p'
!echo '---'
# Where did it land, and what QoS class did it get?
!kubectl get pod sized -o jsonpath='Node: {.spec.nodeName}{"\n"}QoS Class: {.status.qosClass}{"\n"}'

## QoS classes — `Guaranteed`, `Burstable`, `BestEffort`

Kubernetes assigns each pod a **QoS class** based on its resource settings. You don't set the class directly; the API server computes it. The class matters when the node runs low on memory and has to evict.

| QoS class | How a pod qualifies |
|---|---|
| **`Guaranteed`** | *Every* container has `requests` and `limits` set for both CPU **and** memory, with `requests == limits` for each. |
| **`Burstable`** | At least one container has a request or limit set, but the pod doesn't meet the `Guaranteed` criteria. The 80% case. |
| **`BestEffort`** | *No* container in the pod has any `requests` or `limits` set at all. |

### Eviction order under memory pressure

When the kubelet detects memory pressure (free memory below thresholds), it evicts pods in this order:

1. **`BestEffort`** pods first.
2. **`Burstable`** pods next, prioritising those using memory above their `requests`.
3. **`Guaranteed`** pods last — only if there's literally no other choice.

So the cheapest insurance against eviction is to set `requests == limits` on memory. That pod is now `Guaranteed`, and the kubelet won't touch it under normal memory pressure.

### OOMKilled vs Evicted

Two distinct kill mechanisms — keep them straight:

- **OOMKilled** — a *single container* exceeded its memory `limit`. The kernel kills the offending process. Surfaces as `OOMKilled` in the container's last state, and a non-zero `restartCount`.
- **Evicted** — the *whole pod* is removed because the node is under pressure. The pod's phase becomes `Failed` with reason `Evicted`. The pod's controller (Deployment, etc.) creates a replacement somewhere else.

`kubectl describe pod` shows you which one happened in its events. The fix is different for each: OOMKilled means "raise the limit or fix the leak"; Evicted means "set `requests`, pick a less-loaded node, or scale up the cluster.

## `LimitRange` and `ResourceQuota` — namespace governance

Once you have more than one team sharing a cluster, you don't want a pod with `BestEffort` (no limits) hogging a whole node, or a `LOG_LEVEL: debug` test accidentally requesting 100 CPU cores. Two cluster-admin objects govern that.

### `LimitRange` — defaults and per-object caps

A `LimitRange` is namespace-scoped. It does two things:

- **Defaults** — when a pod is created without `requests` or `limits`, the API server fills them in.
- **Caps** — rejects pods whose values are outside an allowed range.

```yaml
apiVersion: v1
kind: LimitRange
metadata:
  name: team-a-defaults
  namespace: team-a
spec:
  limits:
    - type: Container
      default:                # applied when limits aren't set
        cpu: 500m
        memory: 256Mi
      defaultRequest:         # applied when requests aren't set
        cpu: 100m
        memory: 64Mi
      max:                    # reject anything above
        cpu: "4"
        memory: 4Gi
      min:                    # reject anything below
        cpu: 10m
        memory: 16Mi
```

After applying that, every container created in `team-a` without limits gets `500m` / `256Mi`. Containers asking for more than `4` CPU are rejected at creation.

### `ResourceQuota` — namespace totals

A `ResourceQuota` is also namespace-scoped. It caps the *sum* of requests, limits, and object counts across the whole namespace.

```yaml
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-a-budget
  namespace: team-a
spec:
  hard:
    requests.cpu: "10"
    requests.memory: 20Gi
    limits.cpu: "20"
    limits.memory: 40Gi
    pods: "50"
    persistentvolumeclaims: "10"
```

Once the namespace hits 10 cores of CPU requests, the next pod that would push it over is rejected at creation. `kubectl describe quota -n team-a` shows used vs hard.

The CKA tests that you can write both of these and understand the difference: **`LimitRange` shapes individual objects; `ResourceQuota` caps the namespace total.**

## `nodeSelector` — the simplest pin

A `nodeSelector` is a map of labels that must *all* match a node's labels for the pod to be schedulable there.

```yaml
spec:
  nodeSelector:
    disktype: ssd
    gpu: "true"
```

That pod will only schedule on a node that has *both* `disktype=ssd` *and* `gpu=true` labels.

Node labels come from two places:

- **Set by the admin.** `kubectl label node worker-3 disktype=ssd`. Convention.
- **Automatic.** The kubelet sets a few labels on its own node when it registers: `kubernetes.io/hostname`, `kubernetes.io/os`, `kubernetes.io/arch`, `topology.kubernetes.io/zone`, `topology.kubernetes.io/region`. You can rely on these.

### When `nodeSelector` is the right tool

When the rule is **"this pod needs a node that has X."** It's a hard requirement, plain equality, no alternatives. For anything more expressive — set-based selectors, preferred-but-not-required, "anti-X" — reach for node affinity below.

### `nodeName` — the bigger hammer

If you put `spec.nodeName: <literal-node>` on a pod, it bypasses the scheduler entirely. The kubelet on that node either runs it (if it can) or rejects it. Use for static-pod-like cases; almost never in normal manifests.

In [ ]:
# Inspect node labels — kind's node has some built-in ones
!kubectl get nodes --show-labels
!echo '---'
# Label the node ourselves
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  echo "labeling $NODE disktype=ssd" && \
  kubectl label node $NODE disktype=ssd --overwrite
!echo '---'
# A pod that requires disktype=ssd
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: pinned
spec:
  nodeSelector:
    disktype: ssd
  containers:
    - name: app
      image: busybox:1.36
      command: ["sleep", "3600"]
EOF
!sleep 5
!kubectl get pod pinned -o wide

## Node affinity — richer placement against node labels

Node affinity is the grown-up version of `nodeSelector`. Two flavours:

- **`requiredDuringSchedulingIgnoredDuringExecution`** — a hard requirement, like `nodeSelector` but with the full set-based selector language. The pod won't schedule unless it matches.
- **`preferredDuringSchedulingIgnoredDuringExecution`** — a soft preference, weighted. The scheduler tries to honour it but will pick a non-matching node if necessary.

```yaml
spec:
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
          - matchExpressions:
              - key: topology.kubernetes.io/zone
                operator: In
                values: [us-east-1a, us-east-1b]
              - key: gpu
                operator: Exists
      preferredDuringSchedulingIgnoredDuringExecution:
        - weight: 50
          preference:
            matchExpressions:
              - key: instance-type
                operator: In
                values: [m5.xlarge]
```

What this says: the pod **must** land in zone `1a` or `1b` **and** on a node with a `gpu` label (any value); **prefer** `m5.xlarge` instances if available.

### Operators

The selector language is the set-based syntax from notebook 02:

- `In`, `NotIn`, `Exists`, `DoesNotExist`, `Gt`, `Lt` (the last two work on integer-string labels).

`Gt` / `Lt` is the only piece beyond what plain labels can do.

### The "ignored during execution" suffix

The current API only supports the `IgnoredDuringExecution` half — once a pod is placed, the affinity is *not* re-checked at runtime. A future `RequiredDuringExecution` would evict pods when the node's labels change to no longer match; it doesn't exist yet, but the field name leaves room for it.

### When to use which

| If… | Use… |
|---|---|
| Single equality match, one rule | `nodeSelector` |
| Multi-condition, set-based, or `Exists` | `nodeAffinity.required` |
| "Try, but don't refuse to schedule" | `nodeAffinity.preferred` |

## Pod affinity and anti-affinity — placement relative to *other pods*

`nodeSelector` and node affinity match against *node* labels. **Pod affinity** matches against *other pods'* labels — the rule becomes "schedule me near (or away from) pods like this."

```yaml
spec:
  affinity:
    podAntiAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        - labelSelector:
            matchLabels:
              app: web
          topologyKey: kubernetes.io/hostname
```

Read that as: "do not schedule me onto a node that already has a pod with `app=web`." For a Deployment with `replicas: 3` and this anti-affinity, the result is one pod per node — the canonical way to spread replicas across hostnames.

### The `topologyKey` is the whole game

`topologyKey` names a *node label* that defines a "topology domain." The most common values:

- **`kubernetes.io/hostname`** — each node is its own domain. "Spread across hostnames."
- **`topology.kubernetes.io/zone`** — each availability zone is one domain. "Spread across zones."
- **`topology.kubernetes.io/region`** — region-level spreading. Multi-region clusters.

`podAffinity` says "schedule me in the same topology domain as a matching pod." `podAntiAffinity` says "schedule me in a *different* topology domain from a matching pod."

### Required vs preferred — same idea as node affinity

Both `podAffinity` and `podAntiAffinity` come in `required...` and `preferred...` flavours. Prefer the **preferred** kind for spreading replicas — `required` anti-affinity with `topologyKey: hostname` and three replicas on a two-node cluster leaves one pod `Pending` forever.

### When to use it

Real cases:

- **Spread replicas of a Deployment across nodes.** `podAntiAffinity` + `topologyKey: hostname` + match own labels.
- **Co-locate cache with app.** `podAffinity`: schedule the cache pod onto a node that already has a matching app pod.
- **Don't put two databases on the same node.** `podAntiAffinity` between database pods.

Topology spread constraints (later in this notebook) often do the same job more concisely.

In [ ]:
# Anti-affinity that spreads pods one-per-node by hostname.
# On single-node kind, only one replica will land — the second stays Pending.
# That demonstrates how scheduling decisions are surfaced in events.
!cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: Deployment
metadata:
  name: spread
spec:
  replicas: 2
  selector:
    matchLabels:
      app: spread
  template:
    metadata:
      labels:
        app: spread
    spec:
      affinity:
        podAntiAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            - labelSelector:
                matchLabels:
                  app: spread
              topologyKey: kubernetes.io/hostname
      containers:
        - name: app
          image: nginx:alpine
EOF
!sleep 8
!kubectl get pods -l app=spread -o wide
!echo '---'
# Any Pending pods will list the scheduling reason in their events
!PENDING=$(kubectl get pods -l app=spread --field-selector=status.phase=Pending -o jsonpath='{.items[0].metadata.name}' 2>/dev/null); \
  if [ -n "$PENDING" ]; then \
    echo "Pending pod: $PENDING — scheduler events:"; \
    kubectl describe pod $PENDING | grep -A4 Events | tail -6; \
  else \
    echo "(both pods scheduled — multi-node cluster or relaxed constraints)"; \
  fi

## Taints and tolerations — the node-side "go away"

`nodeSelector` and node affinity are *pod-side* — "I want this kind of node." **Taints** are *node-side* — "stay away unless you've explicitly opted in." Together, they let cluster admins reserve nodes for specific workloads.

### Tainting a node

```bash
kubectl taint nodes gpu-node-1 dedicated=gpu:NoSchedule
```

The format is `<key>=<value>:<effect>`. The three effects:

| Effect | Meaning |
|---|---|
| `NoSchedule` | New pods that don't tolerate this taint won't schedule here. Existing pods stay. |
| `PreferNoSchedule` | Same idea, soft. The scheduler tries to avoid, but won't refuse. |
| `NoExecute` | New pods that don't tolerate this taint won't schedule, **and** existing pods that don't tolerate are evicted. Use sparingly. |

### Tolerating on the pod side

```yaml
spec:
  tolerations:
    - key: "dedicated"
      operator: "Equal"
      value: "gpu"
      effect: "NoSchedule"
```

A toleration *unlocks* the matching taint — the pod is now eligible to schedule on the tainted node. **It doesn't force it there.** Tolerations and affinity work together: tolerate the taint *and* use `nodeSelector` / `nodeAffinity` to actually steer the pod onto that node.

`operator: Exists` matches the key regardless of value (the `value` field is omitted). `operator: Equal` (the default) matches both key and value.

### Built-in taints — what the system uses on its own

Kubernetes itself uses taints for several lifecycle conditions. The kubelet automatically adds them:

| Taint | When |
|---|---|
| `node.kubernetes.io/not-ready` | Node's `Ready` condition is `False`. |
| `node.kubernetes.io/unreachable` | Node controller can't reach the node. |
| `node.kubernetes.io/memory-pressure` | Out of memory. |
| `node.kubernetes.io/disk-pressure` | Out of disk. |
| `node.kubernetes.io/unschedulable` | Cordoned (`kubectl cordon`). |
| `node-role.kubernetes.io/control-plane:NoSchedule` | The control-plane taint on managed master nodes. |

Every regular pod implicitly tolerates `not-ready` and `unreachable` for 300 seconds so a brief blip doesn't immediately evict. Set `tolerationSeconds` to change that window — critical for system pods that should stay even when nodes flap.

### Removing a taint

```bash
kubectl taint nodes gpu-node-1 dedicated=gpu:NoSchedule-    # trailing minus removes it
```

The CKA expects you to be quick with both `kubectl taint` and writing tolerations by hand.

In [ ]:
# Pick the first node and taint it
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  echo "tainting node $NODE" && \
  kubectl taint nodes $NODE workload=special:NoSchedule --overwrite
!echo '---'
# A pod that does NOT tolerate the taint — won't schedule
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: untolerated
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sleep", "3600"]
EOF
!sleep 5
!kubectl get pod untolerated
!echo '---'
!kubectl describe pod untolerated | grep -A4 Events | tail -6
!echo '---'
# A pod that DOES tolerate — schedules fine
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: tolerated
spec:
  tolerations:
    - key: workload
      operator: Equal
      value: special
      effect: NoSchedule
  containers:
    - name: app
      image: busybox:1.36
      command: ["sleep", "3600"]
EOF
!sleep 5
!kubectl get pod tolerated
!echo '---'
# Clean up the taint and the stuck pod
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  kubectl taint nodes $NODE workload=special:NoSchedule-
!kubectl delete pod untolerated

## Eviction — what happens when nodes get squeezed

We've touched eviction in passing. Pulling it together.

### Node-pressure eviction

The kubelet monitors **eviction signals** — memory available, root filesystem available, image filesystem available, inode count — and against configured thresholds, it evicts pods to relieve the pressure. The order is:

1. **`BestEffort`** pods first.
2. **`Burstable`** pods exceeding their `requests`, sorted by how much they're over.
3. **`Guaranteed`** pods only as a last resort.

Within each tier, the kubelet picks pods that are using the most of the pressed resource.

### API-initiated eviction

Two patterns:

- **`kubectl drain <node>`** — gracefully evict every non-DaemonSet pod off a node. Used before maintenance. Internally calls the eviction API, which respects `PodDisruptionBudget`.
- **`kubectl delete pod`** — *not* an eviction; just a direct delete. Doesn't respect PDBs.

### `PodDisruptionBudget` — guard against eviction

A `PodDisruptionBudget` (PDB) caps how many pods of a workload can be voluntarily disrupted at once. The drain operation respects it.

```yaml
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: web
spec:
  minAvailable: 2          # or maxUnavailable: 1
  selector:
    matchLabels:
      app: web
```

Match labels of pods (typically via a Deployment's labels). The eviction API refuses to evict pods if doing so would drop the available count below `minAvailable` (or above `maxUnavailable`).

PDBs only constrain **voluntary** disruptions — drain, autoscaling, controlled rollouts. They don't help against node hardware failure (involuntary disruption).

### The big picture

Set `requests` so the scheduler doesn't oversubscribe nodes. Set `limits` so one runaway container can't take down its neighbours. Set `Guaranteed` QoS on truly critical pods. Define PDBs for anything you don't want drained mid-night.

## Topology spread constraints — even distribution as a first-class field

Pod anti-affinity can spread replicas across hostnames or zones, but at scale it gets hard to read and the scheduler can struggle. **Topology spread constraints** are a more direct way to say "spread evenly across this topology, with this much slack."

```yaml
spec:
  topologySpreadConstraints:
    - maxSkew: 1                         # at most a 1-pod imbalance between any two domains
      topologyKey: topology.kubernetes.io/zone
      whenUnsatisfiable: DoNotSchedule   # or ScheduleAnyway
      labelSelector:
        matchLabels:
          app: web
  containers: ...
```

`maxSkew` is the maximum allowed imbalance: the count of matching pods in any one topology domain can differ from the count in any other by at most `maxSkew`. `1` is the tightest, most-common setting.

`whenUnsatisfiable` is the equivalent of "required vs preferred":

- **`DoNotSchedule`** — if the constraint can't be honoured, the pod stays Pending.
- **`ScheduleAnyway`** — if it can't be honoured, schedule anywhere; just prefer balanced placement.

### Default constraints

The cluster's scheduling profile (set by `kube-scheduler` config) may ship default topology constraints — typical defaults spread across `kubernetes.io/hostname` and `topology.kubernetes.io/zone` with `whenUnsatisfiable: ScheduleAnyway`. That's why managed clusters often spread replicas across zones "for free."

### When to use spread vs anti-affinity

- For "spread replicas evenly" — use **topology spread constraints**. They scale better and read more clearly.
- For "don't co-locate me with a *different* workload" — use **pod anti-affinity**. (Spread constraints only know about *one* `labelSelector`.)

## `PriorityClass` and preemption — when high-priority pods kick others out

By default, a Pending pod waits politely until capacity is available. **`PriorityClass`** lets you tag a workload as "important enough that lower-priority pods should be kicked out to make room."

```yaml
apiVersion: scheduling.k8s.io/v1
kind: PriorityClass
metadata:
  name: high-priority
value: 1000000
globalDefault: false
description: "For critical control-plane workloads"
preemptionPolicy: PreemptLowerPriority    # or Never
---
apiVersion: v1
kind: Pod
metadata:
  name: critical
spec:
  priorityClassName: high-priority
  containers: ...
```

When a Pending pod with `high-priority` can't find a node, the scheduler checks: is there a node where *evicting some lower-priority pods* would make this one fit? If so, it evicts them and schedules the high-priority pod. The evicted pods go back to Pending and are rescheduled (if their controller wants them).

### Built-in priority classes

Two ship with every cluster:

- **`system-cluster-critical`** (priority `2000000000`) — for cluster-wide critical workloads (CoreDNS, metrics-server).
- **`system-node-critical`** (priority `2000001000`) — for node-critical workloads (kube-proxy, CNI agents).

Don't use those values for application workloads. Define your own much lower — for example, `1000` for "elevated" and `100` for "background."

### `preemptionPolicy: Never`

A pod with this policy still respects its priority for *queue order* (the scheduler considers higher-priority Pending pods first), but won't *preempt* others. Useful for batch work that should run when capacity exists but shouldn't kick interactive workloads.

## How the scheduler decides — filter, score, profiles

We started the notebook with the scheduler's loop: filter, score, bind. Now the detail.

### Filter (predicates)

A series of **filter plugins** each return yes/no for each node. A node has to pass *all* filters to be a candidate. The standard set:

- **`NodeResourcesFit`** — the pod's `requests` fit in the node's available capacity.
- **`NodeAffinity`** — `nodeSelector` and `nodeAffinity.required` are satisfied.
- **`NodePorts`** — host-network pods don't conflict with existing host ports.
- **`PodTopologySpread`** — spread constraints honoured.
- **`TaintToleration`** — `NoSchedule` and `NoExecute` taints are tolerated.
- **`VolumeBinding`** — bound volumes are available in the node's zone.

### Score (priorities)

A series of **score plugins** assign 0 to 100 to each surviving node. The scheduler sums weighted scores; highest wins. The standard set:

- **`NodeResourcesBalancedAllocation`** — prefer nodes whose CPU and memory utilisation are *balanced*.
- **`NodeResourcesFit`** (score mode) — `LeastAllocated` (default — pack lightly) or `MostAllocated` (bin-pack).
- **`InterPodAffinity`**, **`NodeAffinity`** (preferred parts) — soft affinity preferences.
- **`ImageLocality`** — prefer nodes that already have the image cached.
- **`TaintToleration`** (score) — prefer nodes with fewer `PreferNoSchedule` taints.

### Scheduling profiles

A cluster's `kube-scheduler` runs one or more **profiles** — named configurations of which plugins are enabled and weighted how. The default profile is `default-scheduler`. You can:

- Enable a non-default profile on a particular pod with `spec.schedulerName: <name>`.
- Run *multiple* schedulers in the same cluster, each watching for pods whose `schedulerName` matches.

You won't write a scheduler profile in the CKA. You should know the field exists and that "the scheduler" isn't a single monolith.

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 08:

```bash
kubectl delete pod sized pinned tolerated 2>/dev/null
kubectl delete deployment spread 2>/dev/null
# Remove the labels and taints we added
NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}')
kubectl label node $NODE disktype-
kubectl taint nodes $NODE workload=special:NoSchedule- 2>/dev/null
```

Or, more broadly:

```bash
kubectl delete pod,deployment --all
```